In [1]:
import numpy as np
import json

from sklearn.linear_model import LogisticRegression
from fairlearn.metrics import demographic_parity_difference
from fairlearn.metrics import equalized_odds_difference

In [21]:
def fairness_analysis(name):
    print(f"\nProcessing {name}...")

    X_train = np.load(f"../datasets/processed/{name}_X_train.npy")
    X_test = np.load(f"../datasets/processed/{name}_X_test.npy")
    y_train = np.load(f"../datasets/processed/{name}_y_train.npy")
    y_test = np.load(f"../datasets/processed/{name}_y_test.npy")

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    age = X_test[:, 0]
    age_group = (age > 0).astype(int)

    dpd = demographic_parity_difference(
        y_true=y_test,
        y_pred=y_pred,
        sensitive_features=age_group
    )

    if abs(dpd) < 0.1:
        bias = "Low"
    elif abs(dpd) < 0.2:
        bias = "Moderate"
    else:
        bias = "High"

    print(f"DPD: {dpd}, Bias: {bias}")

    result = {
        "dpd": float(dpd),
        "bias": bias
    }

    with open(f"../fairness/{name}_fairness.json", "w") as f:
        json.dump(result, f)

    print(f"{name} saved")

    y_test = (y_test > 0).astype(int)
    y_pred = (y_pred > 0).astype(int)

    eod = equalized_odds_difference(
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=age_group
    )
    print("EOD:", eod)
    
    result = {
    "dpd": float(dpd),
    "eod": float(eod),
    "bias": bias
}

In [22]:
fairness_analysis("diabetes")
fairness_analysis("heart")
fairness_analysis("liver")
fairness_analysis("kidney")


Processing diabetes...
DPD: 0.6125637231187355, Bias: High
diabetes saved
EOD: 0.5067021242510653

Processing heart...
DPD: 0.19674398566990908, Bias: Moderate
heart saved
EOD: 0.258991008991009

Processing liver...
DPD: 0.021527247558726315, Bias: Low
liver saved
EOD: 0.0

Processing kidney...
DPD: 0.7317073170731707, Bias: High
kidney saved
EOD: 0.0
